# Integre seu API Gateway como um Target do AgentCore Gateway

## Visão Geral

À medida que as organizações exploram as possibilidades de aplicações agênticas, elas continuam enfrentando desafios ao usar dados corporativos como contexto em requisições de invocação a large language models (LLMs) de forma segura e alinhada com as políticas da empresa. Para ajudar a padronizar e proteger essas interações, muitas organizações estão utilizando a especificação Model Context Protocol (MCP), que define como aplicações agênticas podem se conectar de forma segura a fontes de dados e ferramentas.

Embora o MCP tenha sido vantajoso para novos casos de uso, as organizações também enfrentam desafios ao trazer seu portfólio de APIs existentes para a era agêntica. O MCP certamente pode encapsular APIs existentes, mas isso requer trabalho adicional — traduzir requisições de MCP para APIs RESTful, garantir que a segurança seja mantida em todo o fluxo de requisições e aplicar a observabilidade padrão necessária para implantações em produção.

O [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html) agora suporta o [Amazon API Gateway](https://aws.amazon.com/api-gateway/) como target, traduzindo requisições MCP para o AgentCore Gateway (ACGW) em requisições RESTful para o API Gateway (APIGW). Agora você pode expor endpoints de API novos e existentes do APIGW para aplicações agênticas via MCP, com segurança e observabilidade integradas. Este notebook cobre essa nova funcionalidade e mostra como implementá-la.

## O que há de novo

O AgentCore Gateway já suporta múltiplos tipos de target, como funções Lambda, schemas OpenAPI, modelos Smithy, servidores MCP, e agora suporta API Gateway.


![](Images/agent-core-gateway-targets.png)


**Nossos clientes construíram com sucesso ecossistemas extensivos de APIs usando API Gateway, conectando backends de diversas aplicações.** À medida que as empresas avançam em direção à próxima geração de aplicações agênticas, a evolução natural é expor essas APIs existentes e ferramentas de backend para sistemas alimentados por IA, permitindo uma integração perfeita entre a infraestrutura estabelecida e agentes inteligentes modernos.

Atualmente, os clientes seguem um fluxo manual onde exportam suas APIs do APIGW como especificação OpenAPI 3 e, em seguida, adicionam ao ACGW como um target OpenAPI. Esta integração visa simplificar esse processo automatizando a conexão entre APIGW e ACGW.


Com esta integração, os clientes não precisarão mais gerenciar esse processo de exportação/importação manualmente. Um novo tipo de target API_GATEWAY será adicionado ao ACGW. Proprietários de REST APIs podem adicionar sua API como um target do ACGW com poucos cliques no console ou um único comando CLI, expondo assim seus métodos REST API existentes como ferramentas MCP via ACGW. Consumidores de API podem então conectar agentes de IA a essas REST APIs através do Model Context Protocol (MCP) e potencializar seus fluxos de trabalho com integração de IA. Suas aplicações agênticas agora podem se conectar à sua API do APIGW nova ou existente. Hoje, esta integração entre ACGW e APIGW suporta autorização IAM e autorização por API key.

![](Images/agent-core-apigw-target.png)

### Detalhes do Tutorial


| Informação           | Detalhes                                                  |
|:---------------------|:----------------------------------------------------------|
| Tipo de tutorial     | Interativo                                                |
| Componentes AgentCore| AgentCore Gateway, AgentCore Identity                     |
| Framework Agêntico   | Strands Agents                                            |
| Tipo de Gateway Target| API Gateway                                              |
| Agente               | Strands                                                   |
| IdP de Auth de Entrada| Amazon Cognito, mas pode usar outros                     |
| Auth de Saída        | IAM Authorization e API Key                               |
| Modelo LLM           | Anthropic Claude Sonnet 4                                 |
| Componentes do tutorial|Invocar API Gateway via target do AgentCore Gateway       |
| Vertical do tutorial | Cross-vertical                                            |
| Complexidade do exemplo| Fácil                                                   |
| SDK utilizado        | boto3                                                     |

## Arquitetura do Tutorial

Este tutorial serve como um exemplo prático do desafio corporativo mais amplo: **Como integrar a API do API Gateway em uma arquitetura de Gateway centralizada para suas aplicações agênticas de próxima geração.**


## Pré-requisitos

Para executar este tutorial você precisará de:
* Jupyter notebook (kernel Python)
* uv
* Credenciais AWS
* Amazon Cognito

In [ ]:
# Install from the requirements file or pyproject.toml file in current directory
!pip3 install --force-reinstall -U -r requirements.txt --quiet

# Feel free to modify it depending upon the environment you are using. The following command might help.
# !uv pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
import os
# Set AWS credentials if not using SageMaker notebook
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_REGION', 'us-west-2')

In [ ]:
# Import utils
import os
import sys
import importlib

# Get the directory of the current script
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# utils.py is in the same directory as the notebook
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# Import utils and reload to pick up any changes
import utils
importlib.reload(utils)

# Setup logging 
import logging

# Configure logging for notebook environment
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()]
)

# Set specific logger levels
logging.getLogger("strands").setLevel(logging.INFO)

## Configurar pré-requisitos para autorização de entrada e saída

- A autorização de entrada autentica requisições de usuários recebidas.
- A autorização de saída ajuda o ACGW a se conectar de forma segura aos targets do gateway, como um APIGW, em nome do usuário autenticado.

![](Images/agent-core-auth.png)
 
Para API Gateway como target, o ACGW suporta os seguintes tipos de autorização de saída:
-	Sem autorização (não recomendado) – Alguns tipos de target oferecem a opção de ignorar a autorização de saída. Não recomendamos esta opção menos segura.

-	Autorização de saída baseada em IAM – Use a [service role do gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-prerequisites-permissions.html#gateway-service-role-permissions) para autorizar o acesso ao target do gateway com AWS Signature Version 4 (Sig V4).

-	API key – Use a API key configurada pelo API Gateway e configurada com o AgentCore Identity. API Keys criadas via APIGW são mapeadas para APIGW Usage Plans, que ajudam você a monitorar e controlar. Consulte esta [documentação](https://docs.aws.amazon.com/apigateway/latest/developerguide/api-gateway-api-usage-plans.html) para mais detalhes.

Para Autorização de Saída com autorização baseada em IAM, a policy deve incluir a permissão "execute-api:Invoke".

#### Vamos criar rapidamente uma IAM Role para o ACGW assumir. 
**Também usaremos esta IAM Role para autorização baseada em IAM ao chamar nosso APIGW.**

In [ ]:
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-agent-core-gateway-role-for-api-gateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role['Role']['Arn'])

## Configurar Autenticação de Entrada no Gateway com Amazon Cognito

Vamos criar um Cognito User Pool para lidar com autenticação baseada em JWT para requisições recebidas no AgentCore Gateway.

**Componentes Criados:**
- User Pool para gerenciamento de identidade
- Resource Server para escopos OAuth 2.0
- Cliente Machine-to-Machine (M2M) para acesso programático


In [ ]:
# Creating Cognito User Pool 
import os
import boto3

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "invoke",  # Just 'invoke', will be formatted as resource_server_id/invoke
    "ScopeDescription": "Scope for invoking the agentcore gateway"},
]

scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
scopeString = " ".join(scope_names)


cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
gw_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {gw_user_pool_id}")

utils.get_or_create_resource_server(cognito, gw_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(cognito, gw_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names)

print(f"Client ID: {gw_client_id}")

# Get discovery URL
gw_cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{gw_user_pool_id}/.well-known/openid-configuration'
print(gw_cognito_discovery_url)

## Criar o Gateway
Vamos agora criar um Agent Core Gateway

In [ ]:
# CreateGateway with Cognito authorizer. Use the Cognito user pool created in the previous step
import boto3
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
auth_config = {
    "customJWTAuthorizer": { 
        "allowedClients": [gw_client_id],  # Client MUST match with the ClientId configured in Cognito. Example: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": gw_cognito_discovery_url
    }
}
create_response = gateway_client.create_gateway(name='sample-ac-gateway-apigw-demo',
    roleArn=agentcore_gateway_iam_role['Role']['Arn'], # The IAM Role must have permissions to create/list/get/delete Gateway
    protocolType='MCP',
    protocolConfiguration={
        'mcp': {
            'supportedVersions': ['2025-03-26'],
            'searchType': 'SEMANTIC'
        }
    },
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration=auth_config,
    description='AgentCore Gateway with API Gateway target'
)
print(create_response)
# Retrieve the GatewayID used for GatewayTarget creation
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

## Implantar API PetStore de Exemplo com Amazon API Gateway
### Visão Geral da API

Vamos implantar uma REST API de exemplo PetStore demonstrando padrões de autorização com IAM e API Key.

**Endpoints da API:**

| Endpoint | Método | Autorização | Descrição |
|:---------|:-------|:------------|:----------|
| `/pets` | GET | IAM (SigV4) | Listar todos os pets disponíveis |
| `/pets` | POST | IAM (SigV4) | Adicionar um novo pet |
| `/pets/{petId}` | GET | IAM (SigV4) | Obter pet por ID |
| `/orders/{orderId}` | GET | API Key | Obter detalhes do pedido |

**Recursos Criados:**
- REST API com integrações mock
- API Key para o endpoint `/orders`
- Usage Plan com limitação de taxa
- Deployment para o stage `dev`

A especificação OpenAPI está disponível em [AgentCore_Sample_API-dev-oas30-apigateway.json](AgentCore_Sample_API-dev-oas30-apigateway.json).


In [ ]:
# Load OpenAPI definition with API Gateway extensions, import to API Gateway, and deploy
# This uses the OpenAPI document that already has integrations and security configured
api_result = utils.create_and_deploy_api_from_openapi_with_extensions(
    filename='AgentCore_Sample_API-dev-oas30-apigateway.json',
    stage_name='dev',
    description='Initial Deployment'
)

# Extract the API details for use in subsequent cells
api_id = api_result['api_id']
api_key = api_result['api_key_value']
invoke_url=api_result['invoke_url']

print(f"\n{'='*70}")
print(f"API Details for Gateway Target Configuration")
print(f"{'='*70}")
print(f"API ID: {api_id}")
print(f"API Name: {api_result['api_name']}")
print(f"Stage: {api_result['stage_name']}")
print(f"API Key ID: {api_result['api_key_id']}")
print(f"Usage Plan ID: {api_result['usage_plan_id']}")
print(f"{'='*70}")

## Validar o Deployment do API Gateway

Vamos testar todos os endpoints para verificar a configuração e autorização adequadas.

**Nota:** Aguarde 10-15 segundos para que as alterações do API Gateway se propaguem pelas zonas de disponibilidade antes de executar os testes.


In [ ]:
test_results = utils.test_api_gateway_endpoints(
    invoke_url=api_result['invoke_url'],
    api_key=api_result['api_key_value'],
    region=os.environ.get('AWS_DEFAULT_REGION', 'us-west-2')
)

# Display detailed results
print("\nDetailed Test Results:")
for endpoint, result in test_results.items():
    print(f"\n{endpoint}: {result['status']}")
    if 'data' in result:
        print(f"  Data: {result['data']}")
    if 'error' in result:
        print(f"  Error: {result['error']}")



## Proteger a API Key gerada no AgentCore Identity
A seção a seguir cria um novo credential provider de API Key no AgentCore Identity que será usado posteriormente ao criar o target do AgentCore Gateway para o endpoint `/orders`. 

A API Key será armazenada de forma segura no AWS Secrets Manager via AgentCore Identity.


In [ ]:
import boto3
import os
credentialProviderArn=""
try:
    # Create Bedrock Agent Core Control client
    REGION = os.environ['AWS_DEFAULT_REGION']
    bedrock_agent_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
    
    # Create API key credential provider in AgentCore identity
    credential_provider_response = bedrock_agent_client.create_api_key_credential_provider(
        name='sample-api-gateway-key-provider',
        apiKey=api_key,
        tags= {
            'ProjectName':'AgentCore-APIGW-Sample-Application'
        }
    )

    # Store the ARNs in variables
    secretArn = credential_provider_response['apiKeySecretArn']['secretArn']
    credentialProviderArn = credential_provider_response['credentialProviderArn']
    print(f"Credential Provider Arn: {credentialProviderArn}")
    
except KeyError as e:
    print(f"Environment variable or response key missing: {e}")
except Exception as e:
    print(f"An error occurred: {e}")


## Criar o Gateway Target
Para criar um target de API Gateway, você precisa especificar o seguinte como parte da configuração do target:
- **toolFilters**: Use para determinar quais recursos na REST API serão expostos como ferramentas no ACGW. Os filtros também suportam wildcards no filterPath.
- **toolOverrides** (opcional): Use para permitir que os usuários sobrescrevam nomes e descrições das ferramentas. Você deve especificar paths e métodos explícitos. Se não especificado, usará as descrições do APIGW.
- **restApiId**: Use para passar o ID do API Gateway. 
- **stage**: Use para passar o nome do stage implantado do API Gateway

![](Images/agent-core-apigw-target.png)

Abaixo estão alguns exemplos de configurações de target:

### Exemplo 1:

Este expõe "GET & POST /pets", "GET /pets/{petId}" para o gateway e sobrescreve seus nomes e descrições de ferramentas.

In [ ]:
{
  "mcp": {
    "apiGateway": {
      "restApiId": "<api-id>",
      "stage": "<stage>",
      "apiGatewayToolConfiguration": {
        "toolFilters": [
          {
            "filterPath": "/pets",
            "methods": ["GET","POST"]
          },
          {
            "filterPath": "/pets/{petId}",
            "methods": ["GET"]
          }
        ],
      "toolOverrides" : [
          {
             "name": "ListPets",
             "path": "/pets",
             "method": "GET",
             "description":"Retrieves all the available Pets."
         },
         {
              "name": "AddPet",
              "path": "/pets",
              "method": "POST",
               "description":"Add a new pet to the available Pets."
          },
          {
             "path": "/pets/{petId}",
             "method": "GET",
             "name": "GetPetById",
             "description": "Retrieve a specific pet by its ID"
         }
         ]	
      }
    }
  }
}

### Exemplo 2
Este irá expor "GET /pets" mas também "GET /pets/{petId}" ou qualquer coisa sob "/pets". Como toolOverrides não foi especificado, usará a descrição do recurso do API Gateway.

In [ ]:
{
  "mcp": {
    "apiGateway": {
      "restApiId": "<api-id>",
      "stage": "<stage>",
      "apiGatewayToolConfiguration": {
        "toolFilters": [
          {
            "filterPath": "/pets/*",
            "methods": ["GET"]
          }
        ]
      }
    }
  }
}


## Criar Gateway Target 1: Endpoints com Autorização IAM

Este target expõe os endpoints `/pets` e `/pets/{petId}` usando autorização IAM.


In [ ]:
import boto3
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

# creating Gateway Target for '/pets' and '/pets/{petId}' resource with IAM credential provider
create_gateway_target_1_response = gateway_client.create_gateway_target(
    name='api-gateway-target-1',
    gatewayIdentifier=gatewayID,
    targetConfiguration=
            {
            "mcp": {
                "apiGateway": {
                "restApiId": api_id,
                "stage": "dev",
                "apiGatewayToolConfiguration": {
                    "toolFilters": [
                    {
                        "filterPath": "/pets",
                        "methods": ["GET","POST"]
                    },
                    {
                        "filterPath": "/pets/{petId}",
                        "methods": ["GET"]
                    }
                    ],
                    "toolOverrides" : [
                    {
                        "name": "List_Pets",
                        "path": "/pets",
                        "method": "GET",
                        "description":"Retrieves all the available Pets."
                    },
                    {
                        "name": "Add_Pet",
                        "path": "/pets",
                        "method": "POST",
                        "description":"Add a new pet to the available Pets."
                    },
                    {
                        "path": "/pets/{petId}",
                        "method": "GET",
                        "name": "GetPetById",
                        "description": "Retrieve a specific pet by its ID"
                    }
                    ]
                    }
                }
                }
            },
    credentialProviderConfigurations=[
        {
        "credentialProviderType": "GATEWAY_IAM_ROLE"
        }
    ]
)
print(f"Create Gateway Target 1 Response: {create_gateway_target_1_response}")
gateway_target1_id=create_gateway_target_1_response['targetId']

## Criar Gateway Target 2: Endpoints com Autorização por API Key

Este target expõe o endpoint `/orders/{orderId}` usando autorização por API Key do credential provider criado anteriormente.


In [ ]:
# Creating 2nd Gateway Target for /orders/{orderId} resource with API Key credential provider
create_gateway_target_2_response = gateway_client.create_gateway_target(
    name='api-gateway-target-2',
    gatewayIdentifier=gatewayID,
    targetConfiguration=
            {
            "mcp": {
                "apiGateway": {
                "restApiId": api_id,
                "stage": "dev",
                "apiGatewayToolConfiguration": {
                    "toolFilters": [
                    {
                        "filterPath": "/orders/{orderId}",
                        "methods": ["GET"]
                    }
                    ],
                    "toolOverrides" : [
                    {
                        "path": "/orders/{orderId}",
                        "method": "GET",
                        "name": "GetOrderById",
                        "description": "Retrieve a specific order by its ID"
                    }
                    ]
                    }
                }
                }
            },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": credentialProviderArn,
                    "credentialParameterName": "x-api-key", 
                    "credentialLocation": "HEADER" 
                }
            }
        }
    ]
)
print(f"Create Gateway Target 2 Response: {create_gateway_target_2_response}")
gateway_target2_id=create_gateway_target_2_response['targetId']

## Verificar o Status do Gateway Target

Certifique-se de que ambos os targets estejam no estado `READY` antes de prosseguir. Se algum target mostrar status `FAILED`, use a [API get_gateway_target](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agentcore-control/client/get_gateway_target.html) para investigar o motivo da falha.


In [ ]:
list_gateway_targets_response = gateway_client.list_gateway_targets(
    gatewayIdentifier=gatewayID,
    maxResults=10
)
for i, item in enumerate(list_gateway_targets_response['items'], 1):
    print(f"Target {i}:")
    print(f"  Target ID: {item['targetId']}")
    print(f"  Name: {item['name']}")
    print(f"  Status: {item['status']}")
    print()


## Testar Integração com Agente de IA

Usaremos o framework Strands AI para testar a integração completa:
1. Listar as ferramentas disponíveis no gateway
2. Invocar endpoints relacionados a pets (auth IAM)
3. Invocar endpoint de pedidos (auth API Key)


In [ ]:
import json

import requests
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent


def get_token():
    token = utils.get_token(gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION)
    return token['access_token']


def create_streamable_http_transport():
    return streamablehttp_client(
        gatewayURL, headers={"Authorization": f"Bearer {get_token()}"}
    )


client = MCPClient(create_streamable_http_transport)

## The IAM group/user/ configured in ~/.aws/credentials should have access to Bedrock model
yourmodel = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0", # may need to update model_id depending on region
    temperature=0.7,
    max_tokens=500,  # Limit response length
)

with client:
    # Call the listTools
    tools = client.list_tools_sync()
    # Create an Agent with the model and tools

    agent = Agent(model=yourmodel, tools=tools)  
    ## you can replace with any model you like
    
    print("\n" + "="*70)
    print("Test 1: List Available Tools")
    print("="*70)
    agent("List all tools available to you")
    
    print("\n" + "="*70)
    print("Test 2: List All Pets (IAM Auth)")
    print("="*70)
    agent("List all the available pets")
    
    print("\n" + "="*70)
    print("Test 3: Get Pet by ID (IAM Auth)")
    print("="*70)
    agent("Tell me about the pet with petId 3")
    
    print("\n" + "="*70)
    print("Test 4: Get Order Details (API Key Auth)")
    print("="*70)
    agent("When will my order be delivered? My order id is 2")



## Limpeza de Recursos

Execute a célula a seguir para remover todos os recursos criados durante este tutorial:

**Recursos a serem deletados:**
1. AgentCore Gateway e targets
2. AgentCore Identity credential provider
3. API Gateway REST API, API Key e Usage Plan
4. Amazon Cognito User Pool e clients
5. IAM Role 


In [ ]:
# Delete AgentCore Gateway and all targets
print("Step 1: Cleaning up AgentCore Gateway resources...")
agentcore_cleanup = utils.delete_agentcore_gateway_and_targets(
    gateway_id=gatewayID,
    region=REGION
)

# Delete AgentCore Identity Credential Provider
print("\nStep 2: Cleaning up AgentCore Identity Credential Provider...")
credential_cleanup = utils.delete_agentcore_credential_provider(
    credential_provider_arn=credentialProviderArn,
    region=REGION
)

# Delete API Gateway and all related resources
print("\nStep 3: Cleaning up API Gateway resources...")
api_cleanup = utils.delete_api_gateway_and_resources(
    api_id=api_id,
    api_key_id=api_result.get('api_key_id'),
    usage_plan_id=api_result.get('usage_plan_id')
)

# Delete Cognito User Pool
print("\nStep 4: Cleaning up Cognito User Pool...")
cognito_cleanup = utils.delete_cognito_user_pool(
    user_pool_name='sample-agentcore-gateway-pool',
    region=REGION
)

# Delete IAM Role
print("\nStep 5: Cleaning up IAM Role...")
iam_cleanup = utils.delete_iam_role(
    role_name='agentcore-sample-agent-core-gateway-role-role'
)

# Overall Summary
print("\n" + "="*70)
print("Overall Cleanup Summary")
print("="*70)
print(f"AgentCore Gateway Deleted: {'✓' if agentcore_cleanup['gateway_deleted'] else '✗'}")
print(f"AgentCore Targets Deleted: {len(agentcore_cleanup['targets_deleted'])}")
print(f"Credential Provider Deleted: {'✓' if credential_cleanup['credential_provider_deleted'] else '✗'}")
print(f"Cognito User Pool Deleted: {'✓' if cognito_cleanup['user_pool_deleted'] else '✗'}")
print(f"Cognito Clients Deleted: {len(cognito_cleanup['clients_deleted'])}")
print(f"IAM Role Deleted: {'✓' if iam_cleanup['role_deleted'] else '✗'}")
print(f"IAM Policies Deleted: {len(iam_cleanup['policies_deleted'])}")
print(f"API Gateway Deleted: {'✓' if api_cleanup['api_deleted'] else '✗'}")
print(f"API Key Deleted: {'✓' if api_cleanup.get('api_key_deleted') else '✗'}")
print(f"Usage Plan Deleted: {'✓' if api_cleanup.get('usage_plan_deleted') else '✗'}")
print("="*70)

## Resumo
Você testou com sucesso a nova integração entre o AgentCore Gateway e o API Gateway, permitindo que você exponha suas REST APIs existentes como endpoints compatíveis com MCP para suas aplicações agênticas de próxima geração. 